In [2]:
import os
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab7")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

RAW_DIR = "../working_dir/raw"
PARQUET_DIR = "../working_dir/parquet"
STAGING_DIR = f"{PARQUET_DIR}/staging"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 23:16:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 23:16:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# --- Reconstrucción defensiva de df_2025 si no viene del Ejercicio 1 ---
try:
    df_2025
    print("Usando df_2025 ya existente en memoria (salida del Ejercicio 1).")
except NameError:
    print("df_2025 no existe en memoria; reconstruyendo desde el Parquet por período...")
    periodos_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]
    from functools import reduce
    df_2025 = reduce(lambda a, b: a.unionByName(b),
                      [spark.read.parquet(f"{STAGING_DIR}/{p}") for p in periodos_2025])

print("Registros en df_2025 (sin filtrar):", df_2025.count())

df_2025 no existe en memoria; reconstruyendo desde el Parquet por período...


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/opt/app/working_dir/parquet/staging/2025T3.